# 🎬 YouTube Video Downloader & Info Grabber

เรียนรู้การดาวน์โหลดวิดีโอ YouTube และดึงข้อมูลต่างๆ ด้วย Python

**ใช้ library:** `yt-dlp` (ตัวต่อยอดจาก youtube-dl ที่อัพเดทบ่อยและเร็วกว่า)

### สิ่งที่จะได้เรียน:
1. ติดตั้ง yt-dlp
2. ดึงข้อมูลวิดีโอ (ชื่อ, ความยาว, ยอดวิว ฯลฯ)
3. ดึง Thumbnail
4. ดาวน์โหลดวิดีโอ
5. ดาวน์โหลดเฉพาะเสียง (MP3)
6. เลือกคุณภาพวิดีโอ
7. ดาวน์โหลดจาก Playlist
8. ดาวน์โหลด Subtitles

## 1. ตรวจสอบ & ติดตั้ง (yt-dlp + ffmpeg)

> **ffmpeg** จำเป็นสำหรับ: merge video+audio, แปลง MP3, ดาวน์โหลด subtitle

In [1]:
import shutil
import platform
import subprocess

# =============================================
# 1) ตรวจสอบ ffmpeg
# =============================================
ffmpeg_path = shutil.which('ffmpeg')
os_name = platform.system()  # 'Darwin' = macOS, 'Windows', 'Linux'

print(f"💻 OS: {os_name}")
if ffmpeg_path:
    # ดึง version
    result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    version_line = result.stdout.split('\n')[0]
    print(f"✅ ffmpeg พร้อมใช้งาน: {ffmpeg_path}")
    print(f"   {version_line}")
else:
    print("❌ ไม่พบ ffmpeg!")
    print()
    if os_name == 'Darwin':
        print("📦 macOS — ติดตั้งด้วย Homebrew:")
        print("   brew install ffmpeg")
        print()
        print("   (ถ้ายังไม่มี Homebrew: /bin/bash -c \"$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)\")")
    elif os_name == 'Windows':
        print("📦 Windows — เลือกวิธีใดวิธีหนึ่ง:")
        print()
        print("   วิธีที่ 1: winget (แนะนำ)")
        print("   winget install ffmpeg")
        print()
        print("   วิธีที่ 2: choco")
        print("   choco install ffmpeg")
        print()
        print("   วิธีที่ 3: ดาวน์โหลดมือ")
        print("   1. ไปที่ https://www.gyan.dev/ffmpeg/builds/")
        print("   2. โหลด ffmpeg-release-essentials.zip")
        print("   3. แตกไฟล์ → copy ffmpeg.exe ไปที่ C:\\ffmpeg\\bin\\")
        print("   4. เพิ่ม C:\\ffmpeg\\bin\\ เข้า System PATH")
    else:
        print("📦 Linux — ติดตั้งด้วย:")
        print("   sudo apt install ffmpeg    # Ubuntu/Debian")
        print("   sudo dnf install ffmpeg    # Fedora")

print()
print("=" * 50)

# =============================================
# 2) ติดตั้ง yt-dlp
# =============================================
try:
    import yt_dlp
    print(f"✅ yt-dlp พร้อมใช้งาน (version: {yt_dlp.version.__version__})")
except ImportError:
    print("📦 กำลังติดตั้ง yt-dlp...")
    subprocess.check_call(['pip', 'install', 'yt-dlp'])
    import yt_dlp
    print(f"✅ ติดตั้ง yt-dlp สำเร็จ (version: {yt_dlp.version.__version__})")

💻 OS: Darwin
✅ ffmpeg พร้อมใช้งาน: /opt/homebrew/bin/ffmpeg
   ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers

📦 กำลังติดตั้ง yt-dlp...
  Using cached yt_dlp-2026.3.3-py3-none-any.whl.metadata (182 kB)
Using cached yt_dlp-2026.3.3-py3-none-any.whl (3.3 MB)
✅ ติดตั้ง yt-dlp สำเร็จ (version: 2026.03.03)


## 2. Import & ตั้งค่า URL

In [2]:
import json
import os

# 🔗 ใส่ URL วิดีโอที่ต้องการ
VIDEO_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # เปลี่ยน URL ตรงนี้

# 📁 โฟลเดอร์สำหรับเก็บไฟล์
DOWNLOAD_DIR = "downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print(f"📁 โฟลเดอร์: {DOWNLOAD_DIR}")
print(f"🔗 URL: {VIDEO_URL}")

📁 โฟลเดอร์: downloads
🔗 URL: https://www.youtube.com/watch?v=dQw4w9WgXcQ


## 3. ดึงข้อมูลวิดีโอ (Title, Views, Duration ฯลฯ)
ดึงข้อมูลโดย **ไม่ต้องดาวน์โหลด** วิดีโอ

In [3]:
# ดึงข้อมูลวิดีโอ (ไม่ดาวน์โหลด)
opts = {'quiet': True, 'no_warnings': True}

with yt_dlp.YoutubeDL(opts) as ydl:
    info = ydl.extract_info(VIDEO_URL, download=False)

# แสดงข้อมูลหลัก
print(f"📺 ชื่อ: {info['title']}")
print(f"👤 ช่อง: {info.get('channel', 'N/A')}")
print(f"👀 ยอดวิว: {info.get('view_count', 0):,}")
print(f"👍 ไลค์: {info.get('like_count', 'N/A'):,}")
print(f"⏱️ ความยาว: {info.get('duration', 0) // 60} นาที {info.get('duration', 0) % 60} วินาที")
print(f"📅 วันอัพโหลด: {info.get('upload_date', 'N/A')}")
print(f"📝 คำอธิบาย (100 ตัวอักษรแรก): {info.get('description', '')[:100]}...")

📺 ชื่อ: Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)
👤 ช่อง: Rick Astley
👀 ยอดวิว: 1,750,510,988
👍 ไลค์: 18,845,373
⏱️ ความยาว: 3 นาที 33 วินาที
📅 วันอัพโหลด: 20091025
📝 คำอธิบาย (100 ตัวอักษรแรก): The official video for “Never Gonna Give You Up” by Rick Astley. 

Never: The Autobiography 📚 OUT NO...


## 4. ดูข้อมูลทั้งหมดที่ดึงได้ (keys)

In [4]:
# ดู key ทั้งหมดที่ yt-dlp ดึงมาได้
print("🔑 ข้อมูลทั้งหมดที่ดึงได้:")
for key in sorted(info.keys()):
    val = info[key]
    if not isinstance(val, (list, dict)):
        print(f"  {key}: {val}")

🔑 ข้อมูลทั้งหมดที่ดึงได้:
  _format_sort_fields: ('quality', 'res', 'fps', 'hdr:12', 'source', 'vcodec', 'channels', 'acodec', 'lang', 'proto')
  _has_drm: None
  abr: 128.93
  acodec: opus
  age_limit: 0
  aspect_ratio: 1.78
  asr: 48000
  audio_channels: 2
  availability: public
  average_rating: None
  channel: Rick Astley
  channel_follower_count: 4470000
  channel_id: UCuAXFkgsw1L7xaCfnd5JJOw
  channel_is_verified: True
  channel_url: https://www.youtube.com/channel/UCuAXFkgsw1L7xaCfnd5JJOw
  chapters: None
  comment_count: 2400000
  creators: None
  description: The official video for “Never Gonna Give You Up” by Rick Astley. 

Never: The Autobiography 📚 OUT NOW! 
Follow this link to get your copy and listen to Rick’s ‘Never’ playlist ❤️ #RickAstleyNever
https://linktr.ee/rickastleynever

“Never Gonna Give You Up” was a global smash on its release in July 1987, topping the charts in 25 countries including Rick’s native UK and the US Billboard Hot 100.  It also won the Brit Award 

## 5. ดึง Thumbnail (รูปปก)

In [5]:
# แสดง Thumbnail ใน notebook
from IPython.display import Image, display

thumbnail_url = info.get('thumbnail', '')
print(f"🖼️ Thumbnail URL: {thumbnail_url}")

if thumbnail_url:
    display(Image(url=thumbnail_url, width=400))

🖼️ Thumbnail URL: https://i.ytimg.com/vi_webp/dQw4w9WgXcQ/maxresdefault.webp


## 6. ดูคุณภาพวิดีโอที่มี (Available Formats)

In [6]:
# แสดงรายการ format ที่ดาวน์โหลดได้
formats = info.get('formats', [])
print(f"📋 จำนวน format ทั้งหมด: {len(formats)}\n")

print(f"{'ID':<8} {'Extension':<6} {'Resolution':<12} {'Note':<30} {'Filesize':<12}")
print("-" * 70)
for f in formats:
    fid = f.get('format_id', '')
    ext = f.get('ext', '')
    res = f.get('resolution', 'audio')
    note = f.get('format_note', '')[:28]
    size = f.get('filesize') or f.get('filesize_approx') or 0
    size_mb = f"{size / 1024 / 1024:.1f} MB" if size else "N/A"
    print(f"{fid:<8} {ext:<6} {res:<12} {note:<30} {size_mb:<12}")

📋 จำนวน format ทั้งหมด: 31

ID       Extension Resolution   Note                           Filesize    
----------------------------------------------------------------------
sb3      mhtml  48x27        storyboard                     N/A         
sb2      mhtml  80x45        storyboard                     N/A         
sb1      mhtml  160x90       storyboard                     N/A         
sb0      mhtml  320x180      storyboard                     N/A         
139      m4a    audio only   low                            1.2 MB      
249      webm   audio only   low                            1.2 MB      
140      m4a    audio only   medium                         3.3 MB      
251      webm   audio only   medium                         3.3 MB      
160      mp4    256x144      144p                           2.0 MB      
278      webm   256x144      144p                           1.5 MB      
394      mp4    256x144      144p                           1.4 MB      
133      mp4    426x24

## 7. ดาวน์โหลดวิดีโอ (คุณภาพดีสุด)
⚠️ **หมายเหตุ:** ต้องมี `ffmpeg` ติดตั้งเพื่อ merge video+audio → `brew install ffmpeg`

In [7]:
# ดาวน์โหลดวิดีโอคุณภาพดีสุด (video + audio merge)
opts = {
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
    'outtmpl': f'{DOWNLOAD_DIR}/%(title)s.%(ext)s',
    'quiet': False,
}

with yt_dlp.YoutubeDL(opts) as ydl:
    ydl.download([VIDEO_URL])

print("✅ ดาวน์โหลดวิดีโอสำเร็จ!")

[youtube] Extracting URL: https://www.youtube.com/watch?v=dQw4w9WgXcQ
[youtube] dQw4w9WgXcQ: Downloading webpage


[youtube] dQw4w9WgXcQ: Downloading android vr player API JSON


[info] dQw4w9WgXcQ: Downloading 1 format(s): 18
[download] Destination: downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster).mp4
[download] 100% of   11.28MiB in 00:00:01 at 10.41MiB/s  
✅ ดาวน์โหลดวิดีโอสำเร็จ!


## 8. ดาวน์โหลดเฉพาะเสียง (MP3)

In [8]:
# ดาวน์โหลดเฉพาะเสียง แปลงเป็น MP3
opts = {
    'format': 'bestaudio/best',
    'outtmpl': f'{DOWNLOAD_DIR}/%(title)s.%(ext)s',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
    'quiet': False,
}

with yt_dlp.YoutubeDL(opts) as ydl:
    ydl.download([VIDEO_URL])

print("✅ ดาวน์โหลด MP3 สำเร็จ!")

[youtube] Extracting URL: https://www.youtube.com/watch?v=dQw4w9WgXcQ
[youtube] dQw4w9WgXcQ: Downloading webpage


[youtube] dQw4w9WgXcQ: Downloading android vr player API JSON
[info] dQw4w9WgXcQ: Downloading 1 format(s): 251
[download] Destination: downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster).webm
[download] 100% of    3.27MiB in 00:00:01 at 2.35MiB/s   
[ExtractAudio] Destination: downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster).mp3
Deleting original file downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster).webm (pass -k to keep)
✅ ดาวน์โหลด MP3 สำเร็จ!


## 9. ดาวน์โหลดวิดีโอแบบเลือกคุณภาพ (720p, 480p ฯลฯ)

In [9]:
# ดาวน์โหลดวิดีโอ 720p (เปลี่ยนตัวเลขได้: 360, 480, 720, 1080)
QUALITY = 720

opts = {
    'format': f'bestvideo[height<={QUALITY}][ext=mp4]+bestaudio[ext=m4a]/best[height<={QUALITY}]',
    'outtmpl': f'{DOWNLOAD_DIR}/%(title)s_{QUALITY}p.%(ext)s',
    'quiet': False,
}

with yt_dlp.YoutubeDL(opts) as ydl:
    ydl.download([VIDEO_URL])

print(f"✅ ดาวน์โหลด {QUALITY}p สำเร็จ!")

[youtube] Extracting URL: https://www.youtube.com/watch?v=dQw4w9WgXcQ
[youtube] dQw4w9WgXcQ: Downloading webpage


[youtube] dQw4w9WgXcQ: Downloading android vr player API JSON
[info] dQw4w9WgXcQ: Downloading 1 format(s): 398+140
[download] Destination: downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_720p.f398.mp4
[download] 100% of   16.78MiB in 00:00:00 at 22.23MiB/s  
[download] Destination: downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_720p.f140.m4a
[download] 100% of    3.29MiB in 00:00:00 at 12.23MiB/s  
[Merger] Merging formats into "downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_720p.mp4"
Deleting original file downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_720p.f398.mp4 (pass -k to keep)
Deleting original file downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_720p.f140.m4a (pass -k to keep)
✅ ดาวน์โหลด 720p สำเร็จ!


## 10. ดาวน์โหลด Subtitles (คำบรรยาย)

In [10]:
# ดูว่ามี subtitle อะไรบ้าง
subs = info.get('subtitles', {})
auto_subs = info.get('automatic_captions', {})

print(f"📝 Subtitles (manual): {list(subs.keys()) if subs else 'ไม่มี'}")
print(f"🤖 Auto-captions: {list(auto_subs.keys())[:10]}...")  # แสดง 10 ภาษาแรก

📝 Subtitles (manual): ['en', 'de-DE', 'ja', 'pt-BR', 'es-419']
🤖 Auto-captions: ['ab', 'aa', 'af', 'ak', 'sq', 'am', 'ar', 'hy', 'as', 'ay']...


In [11]:
# ดาวน์โหลดวิดีโอพร้อม subtitle (เปลี่ยนภาษาได้: en, th, ja ฯลฯ)
opts = {
    'format': 'best[ext=mp4]/best',
    'outtmpl': f'{DOWNLOAD_DIR}/%(title)s_sub.%(ext)s',
    'writesubtitles': True,        # subtitle ที่คนทำ
    'writeautomaticsub': True,     # auto-generated subtitle
    'subtitleslangs': ['en', 'th'],
    'subtitlesformat': 'srt',
    'quiet': False,
    'skip_download': True,         # ดาวน์โหลดเฉพาะ subtitle (เปลี่ยนเป็น False ถ้าต้องการวิดีโอด้วย)
}

with yt_dlp.YoutubeDL(opts) as ydl:
    ydl.download([VIDEO_URL])

print("✅ ดาวน์โหลด subtitle สำเร็จ!")

[youtube] Extracting URL: https://www.youtube.com/watch?v=dQw4w9WgXcQ
[youtube] dQw4w9WgXcQ: Downloading webpage


[youtube] dQw4w9WgXcQ: Downloading android vr player API JSON
[info] dQw4w9WgXcQ: Downloading subtitles: en, th
[info] dQw4w9WgXcQ: Downloading 1 format(s): 18
[info] Writing video subtitles to: downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_sub.en.srt


[download] Destination: downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_sub.en.srt
[download] 100% of    4.30KiB in 00:00:00 at 30.32KiB/s
[info] Writing video subtitles to: downloads/Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_sub.th.srt


ERROR: Unable to download video subtitles for 'th': HTTP Error 429: Too Many Requests


DownloadError: ERROR: Unable to download video subtitles for 'th': HTTP Error 429: Too Many Requests

## 11. ดึงข้อมูล Playlist ทั้งหมด

In [12]:
# ดึงข้อมูลทุกวิดีโอใน Playlist (ไม่ดาวน์โหลด)
PLAYLIST_URL = "https://www.youtube.com/playlist?list=PLrAXtmErZgOeiKm4sgNOknGvNjby9efdf"  # เปลี่ยน URL

opts = {
    'quiet': True,
    'extract_flat': True,  # ดึงแค่ข้อมูล ไม่โหลดทีละตัว (เร็วมาก)
}

with yt_dlp.YoutubeDL(opts) as ydl:
    playlist_info = ydl.extract_info(PLAYLIST_URL, download=False)

print(f"📋 Playlist: {playlist_info.get('title', 'N/A')}")
print(f"📺 จำนวนวิดีโอ: {len(playlist_info.get('entries', []))}\n")

for i, entry in enumerate(playlist_info.get('entries', [])[:10], 1):
    title = entry.get('title', 'N/A')
    duration = entry.get('duration') or 0
    print(f"  {i}. {title} ({duration // 60}:{duration % 60:02d})")

📋 Playlist: Select Lectures
📺 จำนวนวิดีโอ: 2

  1. Deep Learning State of the Art (2020) (87:41)
  2. Deep Learning Basics: Introduction and Overview (68:06)


## 12. บันทึกข้อมูลวิดีโอเป็น JSON

In [13]:
# บันทึกข้อมูลวิดีโอเป็น JSON
video_data = {
    'title': info['title'],
    'channel': info.get('channel'),
    'upload_date': info.get('upload_date'),
    'duration': info.get('duration'),
    'view_count': info.get('view_count'),
    'like_count': info.get('like_count'),
    'description': info.get('description'),
    'tags': info.get('tags', []),
    'categories': info.get('categories', []),
    'thumbnail': info.get('thumbnail'),
    'url': info.get('webpage_url'),
}

output_path = f"{DOWNLOAD_DIR}/video_info.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(video_data, f, ensure_ascii=False, indent=2)

print(f"✅ บันทึกข้อมูลไปที่ {output_path}")
print(json.dumps(video_data, ensure_ascii=False, indent=2)[:500])

✅ บันทึกข้อมูลไปที่ downloads/video_info.json
{
  "title": "Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)",
  "channel": "Rick Astley",
  "upload_date": "20091025",
  "duration": 213,
  "view_count": 1750510988,
  "like_count": 18845373,
  "description": "The official video for “Never Gonna Give You Up” by Rick Astley. \n\nNever: The Autobiography 📚 OUT NOW! \nFollow this link to get your copy and listen to Rick’s ‘Never’ playlist ❤️ #RickAstleyNever\nhttps://linktr.ee/rickastleynever\n\n“Never Gonna Give You Up” was 


## 13. ดาวน์โหลดพร้อม Progress Hook (แสดง % ดาวน์โหลด)

In [14]:
# ใช้ progress hook เพื่อ track ความก้าวหน้า
def progress_hook(d):
    if d['status'] == 'downloading':
        pct = d.get('_percent_str', 'N/A')
        speed = d.get('_speed_str', 'N/A')
        eta = d.get('_eta_str', 'N/A')
        print(f"\r⬇️ {pct} | Speed: {speed} | ETA: {eta}", end='')
    elif d['status'] == 'finished':
        print(f"\n✅ ดาวน์โหลดเสร็จ! กำลังแปลงไฟล์...")

opts = {
    'format': 'best[ext=mp4][height<=720]/best',
    'outtmpl': f'{DOWNLOAD_DIR}/%(title)s_hook.%(ext)s',
    'progress_hooks': [progress_hook],
    'quiet': True,
    'no_warnings': True,
}

with yt_dlp.YoutubeDL(opts) as ydl:
    ydl.download([VIDEO_URL])

print("✅ เสร็จสมบูรณ์!")

[download] 100% of   11.28MiB in 00:00:00 at 15.37MiB/s    
✅ ดาวน์โหลดเสร็จ! กำลังแปลงไฟล์...
✅ เสร็จสมบูรณ์!                                        


## 14. ดูไฟล์ที่ดาวน์โหลดแล้ว

In [15]:
# แสดงไฟล์ทั้งหมดในโฟลเดอร์ downloads
for f in sorted(os.listdir(DOWNLOAD_DIR)):
    size = os.path.getsize(os.path.join(DOWNLOAD_DIR, f))
    size_mb = size / 1024 / 1024
    print(f"  📄 {f} ({size_mb:.1f} MB)")

  📄 Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster).mp3 (4.9 MB)
  📄 Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster).mp4 (11.3 MB)
  📄 Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_720p.mp4 (20.1 MB)
  📄 Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_hook.mp4 (11.3 MB)
  📄 Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)_sub.en.srt (0.0 MB)
  📄 video_info.json (0.0 MB)


---

## 📌 สรุป Options ที่ใช้บ่อย

| Option | ค่า | คำอธิบาย |
|--------|-----|---------|
| `format` | `best` | คุณภาพดีสุด |
| `format` | `bestaudio/best` | เสียงดีสุด (สำหรับ MP3) |
| `format` | `bestvideo[height<=720]+bestaudio` | จำกัดความสูง 720p |
| `outtmpl` | `%(title)s.%(ext)s` | ตั้งชื่อไฟล์ตามชื่อวิดีโอ |
| `writesubtitles` | `True` | ดาวน์โหลด subtitle |
| `subtitleslangs` | `['en','th']` | เลือกภาษา subtitle |
| `extract_flat` | `True` | ดึงข้อมูล playlist เร็ว (ไม่โหลด) |
| `postprocessors` | FFmpegExtractAudio | แปลงเป็น MP3/AAC |
| `progress_hooks` | `[func]` | ติดตาม % ดาวน์โหลด |
| `skip_download` | `True` | ดึงข้อมูลอย่างเดียว ไม่ดาวน์โหลด |

> ⚠️ **สำคัญ:** ติดตั้ง `ffmpeg` ก่อนใช้งาน → `brew install ffmpeg` (macOS) หรือ `sudo apt install ffmpeg` (Linux)